# 📗 Supabase 와 pgvector — 조건으로 좁히고, 의미로 찾기

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

내 컴퓨터의 파일 하나(sqlite)에서 **클라우드 PostgreSQL(Supabase)** 로 옮겨 갑니다. 그 위에 **pgvector** 를 얹어, 지난 시간에 배운 조건·관계 검색과 **의미 검색**을 한자리에서 만나게 하는 것이 오늘의 전부입니다.

**오늘 이 노트북을 마치면**

- sqlite 와 PostgreSQL 이 어디서 같고 어디서 다른지 짚어 말할 수 있습니다.
- Supabase 프로젝트를 만들고, 표·인덱스·함수를 **SQL Editor** 에서 준비할 수 있습니다.
- **supabase-py 클라이언트**로 행을 넣고·읽고·고치고·지울 수 있습니다.
- 임베딩을 담는 표를 만들어 **벡터 데이터셋을 직접 구축**하고, 분류로 좁힌 의미 검색을 함수 하나로 완성할 수 있습니다.

## ⏪ 복습 — 지난 두 노트북에서

- 표는 **열**과 **행**으로 이뤄지고, **타입**과 **제약조건**이 값의 종류와 범위를 정합니다.
- **기본키**로 행을 집고 **외래키**로 표를 잇습니다. 외래키는 N 쪽에 둡니다.
- `SELECT` · `WHERE` · `ORDER BY` · `GROUP BY` · `HAVING` · `JOIN` · `LEFT JOIN` · 서브쿼리로 원하는 것만 꺼내 리포트를 만들었습니다.
- `UPDATE` · `DELETE` 는 같은 조건으로 `SELECT` 를 먼저 돌리고 `BEGIN`/`ROLLBACK` 으로 감쌌습니다.

여기까지가 **정형 데이터**를 다루는 뼈대입니다. 오늘은 그 뼈대 위에 **의미 검색**을 얹습니다.

## 1. 왜 PostgreSQL 인가 — sqlite 로는 못 한 일들

**옮겨 가는 이유는 아래 표의 세 번째 줄 하나입니다.** 표에 **임베딩(벡터)** 을 함께 담으려면 벡터 타입이 있어야 하는데 sqlite 에는 없습니다.

| | sqlite | PostgreSQL |
|---|---|---|
| 어디에 있나 | 내 컴퓨터의 파일 하나 | 서버(또는 클라우드) |
| 여러 사람이 동시에 | 쓰기는 한 번에 하나만 | 여러 곳에서 동시에 |
| 확장 기능 | 기본 제공 기능만으로는 거의 없음 (FTS5 등 일부 확장은 있습니다) | 필요한 기능을 끼워 넣음(**pgvector** 등) |
| 타입 검사 | `STRICT` 를 붙여야 | 언제나 |
| 외래키 검사 | `pragma` 로 켜야 | 언제나 |

**문법 차이 총정리** — 지금까지 sqlite 로 쓴 것들이 PostgreSQL 에서는 이렇게 바뀝니다.

| 하는 일 | sqlite | PostgreSQL |
|---|---|---|
| 자동 증가 기본키 | `integer PRIMARY KEY AUTOINCREMENT` | `bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY` |
| 글자 | `text` (길이 제한은 `CHECK`) | `text` · `varchar(n)` |
| 소수 | `real` | `numeric` (돈은 이쪽) · `real` |
| 참·거짓 | 0 · 1 | `boolean` |
| 날짜·시각 | `text` 에 ISO 문자열 | `date` · `timestamptz` |
| 타입 검사 | 표에 `STRICT` | 기본 동작 |
| 글자 패턴 검색의 대소문자 | `LIKE` 가 **영문에 한해 무시**(한글 등은 구분) | `LIKE` 는 **구분** — 무시하려면 `ILIKE` |
| 표 비우기 | `DELETE FROM 표` | `TRUNCATE TABLE 표` (훨씬 빠름) |
| 벡터 | (없음) | `vector(n)` + `<=>` (pgvector 확장) |

**바뀌지 않는 것이 훨씬 많습니다.** `SELECT` · `WHERE` · `GROUP BY` · `HAVING` · `JOIN` · `LEFT JOIN` · 서브쿼리 · `INSERT` · `UPDATE` · `DELETE` 는 **그대로**입니다.

## 2. Supabase 시작하기

**Supabase** 는 PostgreSQL 을 클라우드에서 바로 쓰게 해 주는 서비스입니다(무료 요금제로 충분합니다).

**준비 순서**

| 단계 | 할 일 |
|---|---|
| 1) 가입 | [supabase.com](https://supabase.com) 에서 계정을 만듭니다 |
| 2) 프로젝트 생성 | **New project** — Region 은 `ap-northeast-2 (Seoul)`. DB 비밀번호는 오늘 쓰지 않지만 다시 볼 수 없으니 따로 보관하세요 |
| 3) 열쇠 두 개 복사 | **Project Settings → API** 에서 **Project URL** 과 **anon public** 키를 복사합니다 |
| 4) `.env` 작성 | 폴더의 `.env.example` 을 `.env` 로 복사하고 `SUPABASE_URL=` · `SUPABASE_ANON_KEY=` 에 붙여넣습니다 |

**대시보드에서 오늘 쓰는 곳 두 군데**

- **SQL Editor** — SQL 을 브라우저에서 바로 실행합니다. 표·인덱스·함수를 만드는 일은 **여기서만** 합니다(3절).
- **Table Editor** — 만든 표를 눈으로 보고 값을 확인합니다(엑셀처럼).

> **anon key 는 브라우저에 노출돼도 되도록 설계된 공개 키**이고, 이 키로 무엇까지 되는지는 데이터베이스 쪽 정책(RLS)이 정합니다. 그래도 `.env` 에 두세요(`.gitignore` 에 이미 있습니다).

### ✅ 바로 확인 퀴즈

`.env` 에 넣은 `SUPABASE_ANON_KEY` 는 무엇을 하는 열쇠일까요?

- **A.** 데이터베이스 관리자 비밀번호입니다
- **B.** 누구나 가질 수 있는 공개 키이고, 실제 권한은 데이터베이스 정책이 정합니다
- **C.** 프로젝트를 지울 때만 필요합니다
- **D.** 결제 정보를 조회하는 키입니다

<details><summary>정답 보기</summary>

**B** — anon key 는 **브라우저에 노출돼도 되도록** 설계된 공개 키입니다. 그래서 실무에서는 표마다 RLS(행 수준 보안) 정책을 켜서 "이 키로는 무엇까지 되는지"를 데이터베이스가 직접 통제합니다. 관리자 권한 키(service_role)는 절대 브라우저나 저장소에 두지 않습니다.

</details>

## 3. SQL Editor 에서 준비 SQL 실행하기 — 표는 여기서 만듭니다

**supabase-py 는 행을 다루는 도구**라 표·확장·함수를 만드는 일(**DDL**)은 보낼 수 없습니다. 그 일은 **대시보드의 SQL Editor** 에서 합니다.

| 하는 일 | 어디서 |
|---|---|
| 확장 켜기 · 표 만들기 · 인덱스 · 함수 정의 (**DDL**) | **Supabase 대시보드 → SQL Editor** |
| 행 넣기 · 읽기 · 고치기 · 지우기 | 노트북의 `supabase.table(...)` |
| 조건 · 정렬 · 개수 | `.eq()` · `.gte()` · `.order()` · `.limit()` |
| 벡터 유사도 검색 | 미리 만들어 둔 SQL 함수를 `supabase.rpc(...)` 로 호출 |

### 1) Supabase 대시보드 → SQL Editor 를 열고, 아래를 통째로 붙여넣어 **Run** 하세요

왼쪽 메뉴의 **SQL Editor** → **New query** → 아래를 전부 붙여넣고 **Run**(`Ctrl/Cmd + Enter`). 같은 내용이 **`data/setup_supabase.sql`** 에도 있으니 그 파일을 열어 복사해도 됩니다.

```sql
-- day17 실습 준비 SQL — Supabase 대시보드 -> SQL Editor 에 통째로 붙여넣고 Run 하세요.
-- 언제 몇 번을 다시 실행해도 안전합니다(표를 지우고 다시 만듭니다).
-- 출처: 연구비 FAQ 는 한국산업기술기획평가원 '연구비 집행 자주묻는질문'
--       (공공데이터포털, 공공누리 제1유형) 정제본.

-- 1) 확장 켜기 — vector 타입과 거리 연산자(<=>)가 생깁니다.
CREATE EXTENSION IF NOT EXISTS vector;

-- 2) 실습 표 정리 (자식 -> 부모 순서)
DROP TABLE IF EXISTS faq_docs CASCADE;
DROP TABLE IF EXISTS faq_category CASCADE;

-- 3) 분류·담당팀 표 (JOIN 대상)
CREATE TABLE faq_category (
    category  varchar(20) PRIMARY KEY,
    team_name varchar(20) NOT NULL,
    phone     varchar(20)
);

INSERT INTO faq_category (category, team_name, phone) VALUES
    ('RCMS일반',   '고객지원팀', '042-712-9000'),
    ('환경설정',   '고객지원팀', '042-712-9000'),
    ('협약정보',   '협약관리팀', '042-712-9100'),
    ('사용등록',   '집행지원팀', '042-712-9200'),
    ('연구비집행', '집행지원팀', '042-712-9200'),
    ('연구비취소', '집행지원팀', '042-712-9200'),
    ('연구비정산', '정산관리팀', '042-712-9300');

-- 4) FAQ 본문 + 임베딩 표 (768차원 = jhgan/ko-sroberta-multitask)
CREATE TABLE faq_docs (
    faq_id    bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    category  varchar(20) NOT NULL REFERENCES faq_category(category),
    question  text NOT NULL,
    answer    text NOT NULL,
    embedding vector(768)
);

-- 5) 벡터 인덱스 — 코사인 거리(<=>)용 HNSW
CREATE INDEX faq_docs_embedding_idx
    ON faq_docs USING hnsw (embedding vector_cosine_ops);

-- 6) 의미 검색 함수 — 노트북에서 supabase.rpc("match_faq", {...}) 로 부릅니다.
--    filter_category 가 NULL 이면 전체에서, 값이 있으면 그 분류 안에서만 찾습니다.
CREATE OR REPLACE FUNCTION match_faq (
    query_embedding vector(768),
    match_count     int  DEFAULT 3,
    filter_category text DEFAULT NULL
)
RETURNS TABLE (
    faq_id     bigint,
    category   varchar(20),
    question   text,
    answer     text,
    team_name  varchar(20),
    phone      varchar(20),
    similarity float
)
LANGUAGE sql STABLE
AS $$
    SELECT d.faq_id,
           d.category,
           d.question,
           d.answer,
           c.team_name,
           c.phone,
           1 - (d.embedding <=> query_embedding) AS similarity
    FROM faq_docs d
    JOIN faq_category c ON c.category = d.category
    WHERE filter_category IS NULL OR d.category = filter_category
    ORDER BY d.embedding <=> query_embedding
    LIMIT match_count;
$$;
```

### 2) 이 SQL 이 하는 일 — 여섯 덩어리

| | 무엇을 | 왜 |
|---|---|---|
| 1 | `CREATE EXTENSION ... vector` | `vector` 타입과 거리 연산자 `<=>` 가 생깁니다 |
| 2 | `DROP TABLE IF EXISTS ...` | 몇 번을 다시 Run 해도 처음 상태로 돌아가게 합니다 |
| 3 | `faq_category` 표 + 7행 | 분류마다 담당팀·전화가 붙습니다(`JOIN` 대상) |
| 4 | `faq_docs` 표 | 질문·답변 옆에 `embedding vector(768)` 열 하나를 더 둡니다 |
| 5 | HNSW 인덱스 | 벡터가 많아졌을 때 전부 훑지 않고 가까운 곳으로 바로 갑니다 (아래 설명) |
| 6 | `match_faq(...)` 함수 | 의미 검색 SQL 을 **함수로 저장**해 두고 노트북에서 부릅니다 |

**5번의 HNSW 는 벡터를 위한 인덱스입니다** — 지난 노트북에서 `orders(customer_id)` 에 만든 찾아보기의 벡터판이라고 보면 됩니다. 다만 **정확한 답을 보장하지는 않습니다** — 가까운 것을 놓칠 수 있습니다(근사 최근접 이웃). 건수가 적으면 인덱스 없이 전부 비교해도 충분하고, 많아지면 약간의 정확도를 내주고 속도를 얻는 선택입니다.

**6번이 오늘의 핵심입니다.** 벡터 검색 SQL 을 **데이터베이스 안의 함수**로 미리 만들어 두는 것인데, 왜 그래야 하고 파이썬에서 어떻게 부르는지는 7절에서 다룹니다.

> 실습이라 **RLS(행 수준 보안)는 켜지 않습니다** — 대시보드의 "RLS disabled" 경고는 그대로 두세요. 실제 서비스라면 표마다 켜는 것이 필수입니다.

## 4. supabase-py 로 연결하기

이제 노트북에서 붙습니다. 필요한 것은 아까 복사한 **Project URL** 과 **anon key** 두 개뿐입니다.

```python
from supabase import create_client

supabase = create_client(url, key)   # url = Project URL, key = anon public 키
```

아래 준비 셀이 `.env` 에서 두 값을 읽어 `supabase` 를 만들고, 응답을 표로 보여 주는 `to_df()` 도 함께 준비합니다. 값이 없으면 **안내와 함께 멈춥니다**.

In [ ]:
# [제공 코드] — Supabase 연결 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

ROOT = Path(".") if Path("data").is_dir() else Path("..")
load_dotenv(ROOT / ".env")

project_url = os.getenv("SUPABASE_URL")
anon_key = os.getenv("SUPABASE_ANON_KEY")
if not project_url or not anon_key:
    raise RuntimeError(
        "Supabase 연결 정보를 찾지 못했습니다 — SUPABASE_URL / SUPABASE_ANON_KEY 가 비어 있습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) Supabase 대시보드 -> Project Settings -> API 에서\n"
        "     Project URL 과 anon public 키를 복사해 .env 에 붙여넣으세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

supabase = create_client(project_url, anon_key)


def to_df(response):
    """supabase 응답의 .data(딕셔너리 목록)를 pandas DataFrame 으로 바꿉니다."""
    return pd.DataFrame(response.data)


print("Supabase 연결 준비 완료 —", project_url)

3절의 준비 SQL 을 정말 Run 했는지 확인합니다. 안 돼 있으면 여기서 멈추고 무엇을 해야 하는지 알려 줍니다 — **표가 아예 없을 때**와 **표는 있는데 비어 있을 때**를 둘 다 잡습니다.

In [ ]:
# [제공 코드] — 준비 SQL 을 Run 했는지 확인합니다 (안 돼 있으면 여기서 멈춥니다)
try:
    ready = supabase.table("faq_category").select("*").order("category").execute()
except Exception as error:
    raise RuntimeError(
        "faq_category 표를 찾지 못했습니다 — 준비 SQL 을 아직 실행하지 않은 것 같습니다.\n"
        "  Supabase 대시보드 -> SQL Editor 를 열고 data/setup_supabase.sql 을\n"
        "  통째로 붙여넣어 Run 한 뒤, 이 셀을 다시 실행하세요.\n"
        f"  (원래 에러: {error})") from error

if len(ready.data) != 7:
    raise RuntimeError(
        f"faq_category 가 7행이어야 하는데 {len(ready.data)}행입니다 — "
        "SQL Editor 에서 data/setup_supabase.sql 을 처음부터 다시 Run 하세요.")

# 검색 함수(준비 SQL 의 6번)까지 만들어졌는지 — 빈 벡터로 한 번 불러 봅니다.
# 표가 아직 비어 있어도 함수가 있으면 빈 목록이 오고, 함수가 없으면 여기서 에러가 납니다.
try:
    supabase.rpc("match_faq", {"query_embedding": [0.0] * 768,
                                "match_count": 1, "filter_category": None}).execute()
except Exception as error:
    raise RuntimeError(
        "match_faq 함수를 찾지 못했습니다 — 준비 SQL 을 중간까지만 붙여넣은 것 같습니다.\n"
        "  data/setup_supabase.sql 의 6번(CREATE OR REPLACE FUNCTION match_faq ...)까지\n"
        "  포함해 **끝까지** 복사한 뒤 SQL Editor 에서 다시 Run 하세요.\n"
        f"  (원래 에러: {error})") from error

print("준비 확인 완료 — faq_category", len(ready.data), "행 · match_faq 함수 있음")
display(to_df(ready))

## 5. 표 다루기 — SQL 문장 대신 메서드를 잇는다

supabase-py 는 SQL 문장을 문자열로 쓰는 대신 **메서드를 점으로 이어** 같은 일을 합니다. 지난 시간에 배운 SQL 과 한 줄씩 대응합니다.

| 하는 일 | SQL | supabase-py |
|---|---|---|
| 조회 | `SELECT * FROM faq_category` | `supabase.table("faq_category").select("*")` |
| 열 고르기 | `SELECT category, phone` | `.select("category, phone")` |
| 조건 | `WHERE team_name = '집행지원팀'` | `.eq("team_name", "집행지원팀")` |
| 크거나 같음 | `WHERE amount >= 1000` | `.gte("amount", 1000)` |
| 정렬 | `ORDER BY category` | `.order("category")` · 내림차순은 `.order("category", desc=True)` |
| 개수 | `LIMIT 2` | `.limit(2)` |
| 넣기 | `INSERT INTO ... VALUES ...` | `.insert({...})` |
| 고치기 | `UPDATE ... SET ... WHERE ...` | `.update({...}).eq(...)` |
| 지우기 | `DELETE FROM ... WHERE ...` | `.delete().eq(...)` |

**마지막에 `.execute()` 를 붙여야 실제로 요청이 나갑니다.** 그전까지는 "무엇을 할지"를 쌓아 두기만 합니다. 결과는 응답 객체의 **`.data`** 에 **딕셔너리 목록**으로 들어 있습니다.

In [ ]:
# 조회 — SELECT * FROM faq_category 와 같은 일입니다.
response = supabase.table("faq_category").select("*").execute()

print("응답의 .data 는 딕셔너리 목록입니다:")
print(response.data[0])

display(to_df(response))   # 표로 보기 좋게 DataFrame 으로

In [ ]:
# 조건·정렬·개수 — WHERE / ORDER BY / LIMIT 자리에 .eq() · .order() · .limit() 가 옵니다.
response = (supabase.table("faq_category")
            .select("category, team_name, phone")
            .eq("team_name", "집행지원팀")
            .order("category")
            .limit(2)
            .execute())

display(to_df(response))

In [ ]:
# 넣기 -> 고치기 -> 지우기 를 임시 분류 하나로 한 바퀴 돌려 봅니다.
# 먼저 지우고 시작해서 이 셀을 여러 번 실행해도 안전하게 만듭니다.
supabase.table("faq_category").delete().eq("category", "임시분야").execute()

supabase.table("faq_category").insert(
    {"category": "임시분야", "team_name": "임시팀", "phone": "042-000-0000"}
).execute()
print("넣은 뒤:", supabase.table("faq_category").select("*").eq("category", "임시분야").execute().data)

supabase.table("faq_category").update({"phone": "042-999-9999"}).eq("category", "임시분야").execute()
print("고친 뒤:", supabase.table("faq_category").select("*").eq("category", "임시분야").execute().data)

supabase.table("faq_category").delete().eq("category", "임시분야").execute()
print("지운 뒤 남은 행 수:",
      len(supabase.table("faq_category").select("*").eq("category", "임시분야").execute().data))

> **`.update()` 와 `.delete()` 에는 조건을 꼭 붙이세요.** `.eq()` 없이 부르면 표 전체가 바뀌거나 지워집니다 — `WHERE` 를 빠뜨린 것과 같은 사고입니다. 같은 조건으로 `.select()` 를 먼저 돌려 보세요.

### 🖐️ 함께 따라하기 — 새 분류를 넣고·고치고·지우기

위 흐름을 **다른 값**으로 한 번 더 해 보세요.

1. `faq_category` 에 `category='샘플분야'` · `team_name='교육지원팀'` · `phone='042-111-1111'` 한 행을 넣습니다.
2. 그 행의 `phone` 을 `'042-222-2222'` 로 고칩니다.
3. `category` 가 `'샘플분야'` 인 행만 조회해 확인합니다.
4. 확인이 끝나면 그 행을 지웁니다.

**확인 기준**: 3번 조회 결과가 1행이고 `phone` 이 `042-222-2222` 입니다. 4번 뒤에 같은 조회가 0행이 됩니다.

In [ ]:
# '샘플분야' 를 넣고, 전화를 고치고, 조회해 확인한 뒤 지우세요.
# 쓸 도구: supabase.table(...).insert / .update / .select / .delete · .eq() · .execute()

### ✅ 바로 확인 퀴즈

노트북에서 `supabase.table("notes").insert(...)` 로 새 표 `notes` 에 행을 넣으려 하는데 "table not found" 에러가 납니다. 무엇을 해야 할까요?

- **A.** `.insert()` 대신 `.upsert()` 를 씁니다
- **B.** SQL Editor 에서 `CREATE TABLE notes ...` 를 먼저 실행합니다
- **C.** anon key 를 service_role 키로 바꿉니다
- **D.** `create_client` 에 표 이름을 넘깁니다

<details><summary>정답 보기</summary>

**B** — 표를 만드는 일(DDL)은 클라이언트가 하지 못합니다. **SQL Editor** 에서 표를 먼저 만들어야 클라이언트가 그 표에 행을 넣을 수 있습니다.

</details>

## 6. 벡터 데이터셋 구축 — 임베딩을 표에 담는다

임베딩과 유사도 검색은 앞 단원에서 이미 배웠습니다. **오늘 새로운 것은 그 벡터를 따로 두지 않고 표의 한 열(`vector(768)`)로 담는다**는 것 하나입니다 — 그래야 분류 조건·담당팀 `JOIN` 과 한 번에 쓸 수 있습니다.

![벡터 거리와 Top-K](images/벡터거리와_TopK.png)

초록 점 세 개를 뽑는 문장이 곧 `ORDER BY embedding <=> 질문벡터 LIMIT 3` 입니다 — 거리가 작은 순으로 줄 세워 앞에서 세 건만 가져옵니다. 그림의 점선 원이 그 `LIMIT 3` 의 경계입니다(3위 0.516 과 4위 0.552 사이).

- 원본: 연구비 집행 **FAQ 104건** · **7가지 분류**(한국산업기술기획평가원, 공공데이터포털 공공누리 제1유형)
- 채울 표: `faq_docs` — 질문·답변·분류 + `embedding vector(768)` (표는 준비 SQL 이 이미 만들었습니다)

**할 일 두 걸음**: 1) 질문을 임베딩한다 → 2) `.insert()` 로 한꺼번에 넣는다.

In [ ]:
# [제공 코드] — 임베딩 모델 준비 (지난 단원에서 쓴 그 모델입니다 — 실행만 하세요)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jhgan/ko-sroberta-multitask")


def embed(text):
    """질문 한 문장을 768개 숫자(파이썬 리스트)로 바꿉니다."""
    return model.encode([text], normalize_embeddings=True)[0].tolist()


print("임베딩 차원:", model.get_embedding_dimension())

In [ ]:
# 1) 원문을 읽어 질문을 통째로 임베딩합니다. 1~2분쯤 걸립니다.
import csv

with open(ROOT / "data" / "research_faq.csv", encoding="utf-8") as f:
    faq_source = list(csv.DictReader(f))

vectors = model.encode([r["question"] for r in faq_source], normalize_embeddings=True)

records = [
    {"category": r["category"], "question": r["question"],
     "answer": r["answer"], "embedding": vector.tolist()}
    for r, vector in zip(faq_source, vectors)
]

print("보낼 준비가 된 행:", len(records), "건 / 벡터 길이:", len(records[0]["embedding"]))

In [ ]:
# 2) 적재 — 20건씩 나눠 보냅니다(한 번에 보내기엔 요청이 큽니다).
# 먼저 비워서, 이 셀을 다시 실행해도 같은 행이 두 번 쌓이지 않게 합니다.
supabase.table("faq_docs").delete().gte("faq_id", 0).execute()

for start in range(0, len(records), 20):
    supabase.table("faq_docs").insert(records[start:start + 20]).execute()

print("적재 완료")

> **벡터는 그냥 파이썬 리스트로 보냅니다** — PostgreSQL 이 `vector(768)` 로 받습니다.

> **`.delete()` 의 `.gte("faq_id", 0)`** 은 언제나 참인 조건이라 "전부"라는 뜻입니다. `.delete()` 에는 **조건을 반드시 붙이는 습관**을 들이세요 — 조건을 빼면 표 전체가 지워집니다.

In [ ]:
# 검산 — 몇 건이 들어갔고 분류별로 몇 건인지 확인합니다.
loaded = supabase.table("faq_docs").select("faq_id, category").execute().data
print("적재된 행 수:", len(loaded), "/ 보낸 행 수:", len(records))

if len(loaded) != len(records):
    raise RuntimeError(
        f"적재가 끝나지 않았습니다 — 데이터베이스에 {len(loaded)}건뿐입니다. "
        "위 적재 셀을 다시 실행하세요.")

from collections import Counter

counts = Counter(r["category"] for r in loaded)
display(pd.DataFrame(sorted(counts.items()), columns=["category", "건수"]))

### 🖐️ 함께 따라하기 — 원본과 대조해 검산하기

데이터를 넣었으면 **넣은 것이 맞는지** 확인해야 합니다. 방금 읽어 둔 원본 `faq_source` 와 데이터베이스의 `faq_docs` 를 분류별로 대조해 보세요.

1. `faq_source` 에서 분류별 건수를 셉니다(`collections.Counter`).
2. `faq_docs` 를 조회해 같은 것을 셉니다.
3. 분류마다 두 수가 같은지 출력하고, 합계도 견줍니다.

**확인 기준**: 7개 분류가 모두 일치하고, 합이 104 입니다.

In [ ]:
# 원본 faq_source 와 faq_docs 의 분류별 건수를 대조해 일치하는지 확인하세요.
# 쓸 도구: Counter · supabase.table(...).select(...).execute().data

### ✅ 바로 확인 퀴즈

임베딩을 따로 파일에 두지 않고 **표의 한 열**로 담으면 좋은 점은?

- **A.** 벡터를 훨씬 작은 용량으로 저장할 수 있습니다
- **B.** 조건·관계와 한 번의 검색에서 함께 쓸 수 있습니다
- **C.** 임베딩 모델을 더 이상 쓰지 않아도 됩니다
- **D.** 검색 결과가 언제나 정확해집니다

<details><summary>정답 보기</summary>

**B** — 벡터를 따로 보관하면 "이 분야 문서 중에서" 같은 조건을 걸기 어렵습니다. **같은 행**에 두면 분류 조건·담당팀 `JOIN` 과 벡터 검색을 한 번에 할 수 있습니다 — 9절에서 합니다.

</details>

## 7. 의미 검색 — 메서드로는 막히고, 그래서 rpc 로 부른다

"가장 비슷한 K 건"을 찾는 SQL 은 늘 같은 모양입니다. `<=>` 는 pgvector 의 **코사인 거리**(작을수록 가깝다)이고, 사람이 읽기 편하도록 `1 - 거리` 를 유사도로 보여 줍니다.

```sql
SELECT question,
       1 - (embedding <=> '질문벡터') AS similarity
FROM faq_docs
ORDER BY embedding <=> '질문벡터'
LIMIT 3;
```

5·6절에서는 하고 싶은 일을 전부 메서드로 옮길 수 있었습니다 — `WHERE` 는 `.eq()`, `ORDER BY` 는 `.order()`, `LIMIT` 은 `.limit()`. 그런데 위 SQL 은 **옮길 자리가 없습니다.**

`supabase.table(...)` 이 실제로 말을 거는 상대는 **PostgREST** 라는 중간 서버입니다. PostgREST 는 표를 HTTP 로 열어 주면서 `select` · `eq` · `order` · `limit` 같은 **열 이름을 가리키는** 요청만 받습니다. 그런데 우리가 넘겨야 하는 것은 열 이름이 아니라 **질문 벡터라는 768개 숫자 값**이고, 그 값이 `embedding <=> 질문벡터` 라는 **식 안에서 계산에 참여**해야 합니다. PostgREST 요청에는 그런 값을 끼워 넣을 칸 자체가 없습니다. (`.order("embedding")` 는 열 이름만 받으니 "어느 벡터에서 가까운 순"인지 말할 방법이 없습니다.)

**막힌 이유가 곧 해법을 가리킵니다** — 값을 받는 칸이 필요하다면, **값을 받는 칸이 이미 있는 것**을 쓰면 됩니다. 함수의 매개변수가 바로 그 칸입니다.

### `rpc` 란 무엇인가 — 데이터베이스 안의 함수를 이름으로 부르기

`rpc` 는 **Remote Procedure Call**(원격 프로시저 호출)의 줄임말입니다. 말 그대로 **저쪽(데이터베이스)에 미리 만들어 둔 함수를, 이름을 대서 부르는 것**입니다. 우리가 3절에서 SQL Editor 로 만든 `match_faq` 가 바로 그 함수입니다.

즉 오늘의 방식은 이렇습니다 — **복잡한 SQL 은 데이터베이스 안에 함수로 넣어 두고, 파이썬에서는 이름과 인자만 건넨다.** 벡터를 태울 칸이 없던 문제는, 함수의 매개변수 `query_embedding` 이 그 칸이 되어 주면서 풀립니다.

**호출 형태**는 `supabase.rpc("함수이름", {"인자이름": 값}).execute()` 입니다. 첫 인자가 함수 이름, 둘째가 인자 딕셔너리입니다.

```python
supabase.rpc("match_faq", {
    "query_embedding": 질문벡터,      # 768개 숫자 리스트
    "match_count": 3,                 # 몇 건까지
    "filter_category": None,          # None 이면 전체에서
}).execute()
```

**딕셔너리의 키는 3절 준비 SQL 에 적은 매개변수 이름과 글자까지 똑같아야 합니다.** 함수를 만들 때 이렇게 선언해 두었으니까요.

```sql
CREATE OR REPLACE FUNCTION match_faq (
    query_embedding vector(768),
    match_count     int  DEFAULT 3,
    filter_category text DEFAULT NULL
)
```

`filter_category` 는 `None`(SQL 의 `NULL`)이면 전체에서, 값을 주면 그 분류 안에서만 찾습니다 — 9절에서 씁니다. 반환은 다른 요청과 똑같이 **`.data`**, 즉 딕셔너리 목록입니다.

> **키를 잘못 쓰면 "인자 이름이 틀렸다"가 아니라 "함수를 찾지 못했다"는 에러가 납니다.** PostgREST 는 함수를 **이름 + 인자 이름들**의 조합으로 찾기 때문입니다. 위 준비 확인 셀의 `match_faq 함수를 찾지 못했습니다` 안내도 이 성질을 이용한 것입니다.

**정리하면 이렇게 갈립니다.**

| 하고 싶은 일 | SQL 로 쓰면 | supabase-py 로는 |
|---|---|---|
| 조건 조회 | `WHERE category = '연구비'` | `.eq("category", "연구비")` |
| 정렬·개수 | `ORDER BY category` · `LIMIT 3` | `.order("category")` · `.limit(3)` |
| **벡터 거리 정렬** | `ORDER BY embedding <=> 질문벡터` | **`.rpc("match_faq", {...})`** |

**표를 다루는 일은 메서드로, 표만으로 안 되는 일은 데이터베이스 함수로 넣고 `rpc` 로 부릅니다.**

In [ ]:
# FAQ 제목과 글자가 거의 겹치지 않는 질문입니다 — 그래도 뜻으로 찾아옵니다.
question = "연구비 카드로 결제한 금액은 언제 빠져나가나요?"

# 저장해 둔 함수를 부릅니다. 벡터·개수·분류(None 이면 전체)를 인자로 넘깁니다.
response = supabase.rpc("match_faq", {
    "query_embedding": embed(question),
    "match_count": 5,
    "filter_category": None,
}).execute()

display(to_df(response)[["category", "question", "team_name", "similarity"]])

### 🖐️ 함께 따라하기 — 다른 질문으로 Top-3

질문 **`"정산 서류는 어떻게 제출하나요?"`** 로 가장 가까운 FAQ **3건**을 찾으세요.

- `match_faq` 를 `rpc` 로 부르고, 전체에서 찾습니다(`filter_category` 는 `None`).
- 결과에서 `category` · `question` · `similarity` 열만 보이면 됩니다.

**확인 기준**: 3행이 나오고 1위 분류가 **연구비정산** 입니다.

In [ ]:
# 질문 "정산 서류는 어떻게 제출하나요?" 로 가장 가까운 FAQ 3건을 찾으세요.
# 쓸 도구: embed() · supabase.rpc("match_faq", {...}) · .execute() · to_df()

### ✅ 바로 확인 퀴즈

벡터 검색 SQL 을 파이썬에서 문자열로 조립하지 않고 **데이터베이스 안의 함수**로 만들어 두는 이유로 알맞은 것은?

- **A.** supabase-py 로는 SQL 문장을 보낼 수 없어서, 그리고 같은 검색을 여러 앱이 함께 쓸 수 있어서
- **B.** 함수로 만들면 검색 결과가 더 정확해져서
- **C.** 임베딩 모델을 데이터베이스가 대신 돌려 주어서
- **D.** 함수 안에서는 인덱스가 필요 없어서

<details><summary>정답 보기</summary>

**A** — 클라이언트는 행을 다루는 도구라 `ORDER BY ... <=>` 같은 SQL 을 보낼 수 없습니다. 검색 로직을 **한 군데(데이터베이스)** 에 두면 노트북·웹 앱·모바일이 같은 함수를 부르게 되어 고칠 곳도 한 군데뿐입니다.

</details>

## 8. 결합 검색 — 분류로 좁히고, 의미로 정렬

의미 검색만 하면 엉뚱한 분야가 섞입니다("**환경설정 분야에서** 계좌 등록 방법을"). `match_faq` 의 `filter_category` 가 그 일을 합니다 — 함수 한 문장 안에 지난 시간의 조건·관계와 오늘의 의미가 함께 있습니다.

| 무엇으로 | 함수 안의 SQL | 하는 일 |
|---|---|---|
| 조건 | `WHERE filter_category IS NULL OR d.category = filter_category` | 분야로 후보를 먼저 좁힙니다 |
| 관계 | `JOIN faq_category c ON c.category = d.category` | 담당팀·전화를 함께 붙입니다 |
| 의미 | `ORDER BY d.embedding <=> query_embedding` | 남은 후보를 뜻이 가까운 순으로 세웁니다 |
| 개수 | `LIMIT match_count` | 위에서 몇 건만 |

In [ ]:
mixed_question = "계좌 등록은 어떻게 하나요?"
mixed_vector = embed(mixed_question)

# 1) 환경설정 분야 안에서만
narrowed = supabase.rpc("match_faq", {
    "query_embedding": mixed_vector,
    "match_count": 3,
    "filter_category": "환경설정",
}).execute()
display(to_df(narrowed)[["category", "question", "team_name", "phone", "similarity"]])

In [ ]:
# 2) 조건을 빼면 무엇이 달라지는지 나란히 봅니다.
everywhere = supabase.rpc("match_faq", {
    "query_embedding": mixed_vector,
    "match_count": 3,
    "filter_category": None,
}).execute()
display(to_df(everywhere)[["category", "question", "similarity"]])

print("조건을 빼면 다른 분야의 FAQ 가 섞여 올라옵니다 — "
      "의미만으로는 '어느 분야 질문인지'를 구별하지 못하기 때문입니다.")

### ✅ 바로 확인 퀴즈

FAQ 104건 중 `category = '환경설정'` 인 것은 10건입니다. `filter_category="환경설정"` · `match_count=3` 으로 부르면 결과는 몇 행일까요?

- **A.** 104행
- **B.** 10행
- **C.** 3행
- **D.** 0행

<details><summary>정답 보기</summary>

**C** — 조건으로 10건까지 좁힌 뒤, 그 안에서 가까운 순으로 세워 위에서 **3건**만 자릅니다. 후보가 3건보다 적으면 그만큼만 나옵니다.

</details>

## 🚀 응용 클론코딩 — FAQ 검색 함수 완성하기

지금까지 만든 것을 함수 하나로 묶습니다. 실무에서 검색 기능은 늘 이 모양입니다.

**요구사항**: `search_faq(question, category=None, k=3)` 를 완성하세요.

1. `question` 을 `embed()` 로 벡터(숫자 리스트)로 바꿉니다.
2. `match_faq` 를 `rpc` 로 부릅니다 — `match_count` 에 `k`, `filter_category` 에 `category`.
3. 응답을 `to_df()` 로 DataFrame 으로 바꿔 `question` · `answer` · `team_name` · `phone` · `similarity` 열만 돌려줍니다.

**확인 기준**

- `search_faq("연구비 카드로 결제한 금액은 언제 빠져나가나요?")` 의 1위 질문에 **'카드'** 가 들어 있습니다.
- `search_faq("계좌 등록은 어떻게 하나요?", category="환경설정")` 는 담당팀이 한 팀으로만 나옵니다.
- `k=5` 를 주면 5행이 나옵니다.

In [ ]:
# search_faq(question, category=None, k=3) 을 완성하고, 위 확인 기준 세 가지를 실행해 보세요.
# 쓸 도구: embed() · supabase.rpc("match_faq", {...}) · .execute() · to_df()

## 오늘 배운 것 · 치트시트

**어디서 무엇을 하나**

| 하는 일 | 어디서 |
|---|---|
| 확장·표·인덱스·함수 만들기 (DDL) | Supabase 대시보드 → **SQL Editor** |
| 행 다루기 | 노트북의 `supabase.table(...)` |
| 벡터 유사도 검색 | 저장해 둔 SQL 함수를 `supabase.rpc(...)` 로 |

**SQL ↔ supabase-py**

| SQL | supabase-py |
|---|---|
| `SELECT * FROM t` | `supabase.table("t").select("*").execute()` |
| `SELECT a, b FROM t` | `.select("a, b")` |
| `WHERE a = 1` | `.eq("a", 1)` |
| `WHERE a >= 1` | `.gte("a", 1)` |
| `ORDER BY a DESC` | `.order("a", desc=True)` |
| `LIMIT 3` | `.limit(3)` |
| `INSERT INTO t ... VALUES ...` | `.insert({...})` · 여러 건은 `.insert([{...}, {...}])` |
| `UPDATE t SET ... WHERE ...` | `.update({...}).eq(...)` |
| `DELETE FROM t WHERE ...` | `.delete().eq(...)` |
| (결과 꺼내기) | `.execute().data` — 딕셔너리 목록 |

**pgvector**

| 목적 | 문법 |
|---|---|
| 확장 켜기 | `CREATE EXTENSION IF NOT EXISTS vector;` |
| 벡터 열 | `embedding vector(768)` |
| 벡터 넣기 | 파이썬 리스트를 그대로 `{"embedding": [0.1, 0.2, ...]}` |
| 거리 | `<=>`(코사인) · `<->`(L2) · `<#>`(음의 내적) — 작을수록 가깝다 |
| 유사도 | `1 - (embedding <=> query_embedding)` |
| Top-K | `ORDER BY embedding <=> query_embedding LIMIT k` |
| 인덱스 | `CREATE INDEX ... USING hnsw (embedding vector_cosine_ops)` |
| 검색 함수 호출 | `supabase.rpc("match_faq", {...}).execute()` |

## ⏭️ 예고 — 다음 단원: LangChain 기본 구조

오늘 만든 `search_faq` 는 사실 **RAG 의 검색 단계**입니다. 여기서 찾아온 근거를 LLM 에 넘기면 출처가 붙은 답변이 됩니다. 과제 LV3 에서 그 마지막 한 걸음을 직접 만들고, 다음 시간부터는 그 흐름을 **LangChain** 으로 조립합니다.